# Nettoyage des données du défi 1 - Décrochage scolaire

Ce notebook construit **un seul fichier CSV de travail**, au grain `province × année scolaire`. Il reste volontairement centré sur le décrochage scolaire. Les fermetures d'écoles, les PDI et la pauvreté seront ajoutées plus tard comme variables explicatives facultatives, jamais comme un autre défi.

Les données sont lues et transformées localement. Le classeur original n'est jamais modifié.

## 1. Sources utilisées

- Classeur principal DGESS 2016-2025 : taux d'achèvement régional.
- Annuaires CNS 521 à 530 : taux provinciaux de promotion, redoublement et abandon.
- Référentiel historique : rattachement des 45 provinces aux régions anciennes et aux régions 2025.

L'achèvement reste explicitement régional : il n'est pas présenté comme une mesure provinciale.

In [ ]:
from pathlib import Path
import json
import sys

RACINE = Path.cwd()
if RACINE.name == '02_nettoyage':
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE / 'scripts'))

from construire_fichier_decrochage import construire, trouver_classeur

CLASSEUR = trouver_classeur(RACINE, None)
ANNUAIRES = RACINE / '01_collecte/data_pack_selection/data/01_Education_MENAPLN_DGESS/CNS_DGESS/Annuaires'
REFERENTIEL = RACINE / 'config/referentiel_provinces_historiques.json'
SORTIE = RACINE / '02_nettoyage/SCO-RISKBurkina_02-nettoyage_D1_engineer_donnees-decrochage-province-annee_v1.csv'
QUALITE = RACINE / 'rapports_prives/05_qualite_fichier_decrochage.json'

print('Classeur trouvé :', CLASSEUR.name)
print('Annuaires disponibles :', len(list(ANNUAIRES.glob('CNS*.csv'))))

## 2. Règles de nettoyage

1. Normaliser les accents, espaces et variantes de noms uniquement pour les jointures.
2. Conserver une graphie canonique lisible dans la sortie.
3. Retirer les lignes régionales et nationales des tables provinciales.
4. Conserver les cinq taux d'abandon par niveau et les six taux de redoublement.
5. Calculer une moyenne provinciale **non pondérée**, clairement nommée comme telle, faute de dénominateurs par niveau dans ce tableau.
6. Joindre l'achèvement au niveau de l'ancienne région sans inventer une valeur provinciale.
7. Conserver les valeurs source hors de 0-100 pour la traçabilité, mais les exclure des moyennes et les signaler.
8. Vérifier l'unicité de la clé `province + année scolaire`.

In [ ]:
rapport = construire(
    classeur=CLASSEUR,
    dossier_annuaires=ANNUAIRES,
    referentiel=REFERENTIEL,
    sortie_csv=SORTIE,
    sortie_qualite=QUALITE,
)

# Le rapport ne contient aucune valeur éducative, seulement les contrôles de structure.
rapport

## 3. Revue humaine du fichier produit

Cette étape affiche les premières lignes uniquement sur votre ordinateur. Vérifiez les noms, les années, les taux et les sources avant de continuer.

In [ ]:
import pandas as pd

df = pd.read_csv(SORTIE)
print('Dimensions :', df.shape)
print('Provinces :', df['province'].nunique())
print('Années :', sorted(df['annee_scolaire'].unique()))
df.head(10)

In [ ]:
# Contrôles essentiels : une ligne par province et année, taux entre 0 et 100.
assert not df.duplicated(['province', 'annee_scolaire']).any(), 'Clés dupliquées'
for colonne in [
    'taux_abandon_moyen_non_pondere',
    'taux_redoublement_moyen_non_pondere',
    'taux_achevement_region',
]:
    valeurs = df[colonne].dropna()
    assert valeurs.between(0, 100).all(), f'Valeur hors limites dans {colonne}'

print('Lignes à vérifier dans la publication source :', int((df['statut_qualite'] != 'ok').sum()))

controle = df.groupby('annee_scolaire').agg(
    provinces=('province', 'nunique'),
    abandon_manquant=('taux_abandon_moyen_non_pondere', lambda s: s.isna().sum()),
    achevement_regional_manquant=('taux_achevement_region', lambda s: s.isna().sum()),
)
controle

## 4. Résultat et limite à documenter

Le fichier `02_nettoyage/SCO-RISKBurkina_02-nettoyage_D1_engineer_donnees-decrochage-province-annee_v1.csv` devient notre source de travail unique. Les moyennes provinciales d'abandon et de redoublement sont des moyennes non pondérées des taux totaux par niveau. Cette limite sera testée et expliquée. La prochaine étape calculera l'indice de risque, sa sensibilité et une prévision de référence avant tout modèle complexe.